# Laboratorio: subespacios y aproximación de rango bajo

Verificaremos el teorema de Eckart-Young-Mirsky, proyectaremos datos sobre su mejor subespacio y estudiaremos una compresión matricial usando una fotografía suministrada por el docente y almacenada dentro del libro.

## 1. Herramientas

La función siguiente devuelve la SVD truncada y controla que el rango solicitado sea válido.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def svd_truncada(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    if not 0 <= k <= len(s):
        raise ValueError('k debe estar entre 0 y min(m,n)')
    Ak = (U[:, :k] * s[:k]) @ Vt[:k, :]
    return Ak, U, s, Vt

def energia_acumulada(s):
    return np.cumsum(s**2) / np.sum(s**2)

## 2. Errores exactos de truncamiento

Construimos una matriz cuyos valores singulares son $6$, $3$ y $1$. Las multiplicaciones ortogonales cambian las entradas, pero no esos valores singulares.

In [ ]:
theta = np.pi/5
V = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0, 0, 1]])
U = np.array([[1, 0, 0, 0],
              [0, 0, 1, 0],
              [0, 1, 0, 0],
              [0, 0, 0, 1]], dtype=float)
Sigma = np.zeros((4, 3))
Sigma[:3, :3] = np.diag([6., 3., 1.])
A = U @ Sigma @ V.T
_, _, s, _ = svd_truncada(A, 3)
assert np.allclose(s, [6, 3, 1])
A

In [ ]:
tabla = []
for k in range(4):
    Ak, _, s, _ = svd_truncada(A, k)
    error_F = np.linalg.norm(A-Ak, 'fro')
    error_2 = np.linalg.norm(A-Ak, 2)
    teorico_F = np.sqrt(np.sum(s[k:]**2))
    teorico_2 = s[k] if k < len(s) else 0.0
    assert np.allclose(error_F, teorico_F)
    assert np.allclose(error_2, teorico_2)
    tabla.append((k, np.linalg.matrix_rank(Ak, tol=1e-10), error_F, error_2))
tabla

## 3. Mejor subespacio para las filas

Tomamos observaciones en $\mathbb R^3$ y calculamos el mejor plano que pasa por el origen. Sus direcciones son los dos primeros vectores singulares derechos.

In [ ]:
X = np.array([[ 2.0,  1.0,  0.2],
              [ 1.2,  0.7, -0.1],
              [-1.8, -0.8,  0.1],
              [ 0.2,  1.8,  0.3],
              [-0.3, -1.5, -0.2],
              [ 1.1, -0.6,  0.4]])
U_X, s_X, Vt_X = np.linalg.svd(X, full_matrices=False)
Vk = Vt_X[:2, :].T
P = Vk @ Vk.T
X2 = X @ P
assert np.allclose(P, P.T) and np.allclose(P@P, P)
assert np.linalg.matrix_rank(X2) <= 2
Vk, X2

La suma de distancias cuadradas debe coincidir con la energía singular descartada. También comparamos la energía capturada con muchos planos aleatorios; esta exploración ilustra el teorema, pero no lo reemplaza.

In [ ]:
error_subespacio = np.linalg.norm(X-X2, 'fro')**2
error_teorico = np.sum(s_X[2:]**2)
assert np.allclose(error_subespacio, error_teorico)

rng = np.random.default_rng(2026)
energias = []
for _ in range(3000):
    Q, _ = np.linalg.qr(rng.normal(size=(3, 2)))
    energias.append(np.linalg.norm(X@Q, 'fro')**2)
energia_optima = np.sum(s_X[:2]**2)
assert energia_optima + 1e-10 >= max(energias)
error_subespacio, energia_optima, max(energias)

## 4. Fotografía para la aplicación

La fotografía se distribuye con el libro en una versión reducida, en escala de grises y sin metadatos EXIF. El código localiza el archivo tanto al ejecutar desde la raíz del libro como desde la carpeta de la unidad.

In [ ]:
candidatos = [Path('figuras/ojo_svd.jpg'),
              Path('unidad_4/figuras/ojo_svd.jpg')]
ruta_imagen = next((p for p in candidatos if p.exists()), None)
if ruta_imagen is None:
    raise FileNotFoundError('No se encontró figuras/ojo_svd.jpg')

with Image.open(ruta_imagen) as im:
    imagen = np.asarray(im.convert('L'), dtype=float)/255.0
m, n = imagen.shape
plt.figure(figsize=(7, 4))
plt.imshow(imagen, cmap='gray', vmin=0, vmax=1)
plt.title('Matriz de intensidades original')
plt.axis('off');

## 5. Reconstrucciones con diferentes rangos

Cada reconstrucción usa los primeros $k$ términos de la expansión SVD.

In [ ]:
U_img, s_img, Vt_img = np.linalg.svd(imagen, full_matrices=False)
rangos = [5, 20, 50, 100]
fig, axes = plt.subplots(1, len(rangos), figsize=(15, 4))
reconstrucciones = {}
for ax, k in zip(axes, rangos):
    Ik = (U_img[:, :k]*s_img[:k]) @ Vt_img[:k, :]
    reconstrucciones[k] = Ik
    error = np.linalg.norm(imagen-Ik, 'fro')/np.linalg.norm(imagen, 'fro')
    ax.imshow(np.clip(Ik, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'k={k}; error={error:.3f}')
    ax.axis('off')
plt.tight_layout();

## 6. Espectro, energía y elección de rango

La energía usa $\sigma_j^2$. Como la intensidad media concentra una parte grande de la energía de una imagen, examinamos umbrales altos y los contrastamos con el error y la apariencia.

In [ ]:
energia = energia_acumulada(s_img)
k99 = int(np.searchsorted(energia, 0.99) + 1)
k999 = int(np.searchsorted(energia, 0.999) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(np.arange(1, len(s_img)+1), s_img, 'o-', ms=3)
axes[0].set(xlabel='índice', ylabel='valor singular', title='Decaimiento singular')
axes[0].grid(alpha=.25)
axes[1].plot(np.arange(1, len(s_img)+1), energia)
axes[1].axhline(.99, color='tab:orange', ls='--')
axes[1].axhline(.999, color='tab:red', ls='--')
axes[1].set(xlabel='rango k', ylabel='energía acumulada', ylim=(0, 1.01),
            title=f'k99={k99}, k99.9={k999}')
axes[1].grid(alpha=.25)
plt.tight_layout();

## 7. Error y almacenamiento nominal

Comparamos el error relativo con la fracción $k(m+n+1)/(mn)$ de números almacenados. El cálculo no representa todavía el tamaño real de un archivo.

In [ ]:
ks = np.arange(1, min(m, n)+1)
errores = np.sqrt(np.array([np.sum(s_img[k:]**2) for k in ks])) / np.linalg.norm(imagen, 'fro')
fraccion = ks*(m+n+1)/(m*n)
assert np.all(np.diff(errores) <= 1e-12)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(ks, errores, color='tab:blue', label='error relativo')
ax1.set(xlabel='rango k', ylabel='error relativo de Frobenius')
ax2 = ax1.twinx()
ax2.plot(ks, fraccion, color='tab:orange', label='almacenamiento nominal')
ax2.set_ylabel('fracción de números almacenados')
ax1.grid(alpha=.25)
plt.title('Calidad frente a almacenamiento')
plt.tight_layout();

## 8. Verificación directa de las dos proyecciones

La misma matriz truncada se obtiene proyectando las filas con $V_kV_k^T$ o las columnas con $U_kU_k^T$.

In [ ]:
k = 20
Uk = U_img[:, :k]
Vk = Vt_img[:k, :].T
Ik_svd = (Uk*s_img[:k]) @ Vk.T
Ik_filas = imagen @ Vk @ Vk.T
Ik_columnas = Uk @ Uk.T @ imagen
assert np.allclose(Ik_svd, Ik_filas)
assert np.allclose(Ik_svd, Ik_columnas)
np.linalg.matrix_rank(Ik_svd, tol=1e-10)

## 9. Actividades para completar

1. Encuentra el menor $k$ que logre error relativo menor que $0.05$.
2. Determina para qué rangos el conteo $k(m+n+1)$ es menor que $mn$.
3. Añade ruido aleatorio a la imagen y estudia cómo cambia el decaimiento singular.
4. Construye una matriz con $\sigma_k=\sigma_{k+1}$ y explora la falta de unicidad del mejor subespacio.
5. Explica por qué el mejor subespacio de los datos sin centrar no es todavía PCA.